# Dataset audit and baseline run notebook (HAM10000) - single run
**How to use:** Runtime > **Change runtime type** > **T4 GPU** > Save. Then Runtime > **Run all**.
A popup asks to connect Google Drive: click **Allow**. Then leave the tab open and the laptop awake. Total time about 1.5-2.5 hours.
If any cell shows red text or `STOP:`, the run stops by itself. Do not retry; send the message to the author of the kit.

Needed in Google Drive under `MyDrive/stage2/`: `HAM10000_images_part_1.zip`, `HAM10000_images_part_2.zip`, `HAM10000_metadata`, and `kit_v2.zip`.

In [ ]:
# CELL 1 (1-2 min) mount Drive, install libraries, show GPU
from google.colab import drive
drive.mount('/content/drive')
!pip -q install scikit-learn pandas matplotlib pillow scipy
!nvidia-smi | head -5

In [ ]:
# CELL 2 (3-6 min) fresh start: remove old outputs, unpack kit and data
!rm -rf /content/out /content/kit
!mkdir -p /content/kit /content/data/part_1 /content/data/part_2
!unzip -o -q /content/drive/MyDrive/stage2/kit_v2.zip -d /content/kit
!unzip -n -q /content/drive/MyDrive/stage2/HAM10000_images_part_1.zip -d /content/data/part_1
!unzip -n -q /content/drive/MyDrive/stage2/HAM10000_images_part_2.zip -d /content/data/part_2
!cp /content/drive/MyDrive/stage2/HAM10000_metadata* /content/data/HAM10000_metadata.csv
!ls /content/kit
!ls /content/data/part_1 | head -3

In [ ]:
# CELL 3 (5-15 min) audit. Final line: 'Audit done'
%cd /content/kit
!python 01_audit.py --meta /content/data/HAM10000_metadata.csv --img_dirs /content/data/part_1 /content/data/part_2 --out /content/out --near_dup_dist 2 --merge_dist 0

In [ ]:
# CELL 4 (under 1 min) leakage-safe splits. Final line: 'OK: zero shared lesion groups'
!python 02_split.py --out /content/out --seeds 42 43 44 45 46

In [ ]:
# CELL 5 (15-40 min) metadata baselines. Final line: seed 46 hgb
!python 03_metadata.py --out /content/out --seeds 42 43 44 45 46 --n_jobs 2

In [ ]:
# CELL 6 (20-60 min) image baseline. Final line: 'done. best val PR-AUC'
!python 04_image.py --out /content/out --seed 42

In [ ]:
# CELL 7 (1 min) safety backup of everything so far to Drive
!rsync -a --exclude='cache_*' --exclude='*.pt' /content/out/ /content/drive/MyDrive/stage2/out_backup_v2/
!ls /content/drive/MyDrive/stage2/out_backup_v2

In [ ]:
# CELL 8 (3-10 min) evaluation and figures. Final line: 'Results and figures written'
!python 05_evaluate.py --out /content/out --seed 42 --extra_seeds 43 44 45 46

In [ ]:
# CELL 9 (1-2 min) near-duplicate sensitivity analysis. Final line: 'Sensitivity analysis written'
!python 06_sensitivity.py --out /content/out --seed 42

In [ ]:
# CELL 10 (1-2 min) final backup and results zip for the author
!rsync -a --exclude='cache_*' --exclude='*.pt' /content/out/ /content/drive/MyDrive/stage2/out_backup_v2/
!rm -f /content/drive/MyDrive/stage2/stage2_results_v2.zip
!cd /content && zip -qr /content/drive/MyDrive/stage2/stage2_results_v2.zip out -x 'out/cache_*' -x 'out/*.pt'
!ls -la /content/drive/MyDrive/stage2/stage2_results_v2.zip
print('ALL DONE - download stage2_results_v2.zip from Drive > stage2')